In [30]:
import pandas as pd
import math
import nupack
from collections import Counter
import DNAFMIndex

Sequence-based Features

In [31]:
def gc_content(seq:str) -> float:
    gc = (seq.count('G') + seq.count('C'))
    return round(gc / len(seq), 3)

def shannon_entropy(seq:str) -> float:
    seq = seq.upper()
    bases = Counter([tmp_base for tmp_base in seq])

    shannon_entropy_value = 0
    for base in bases:
        n_i = bases[base]
        p_i = n_i / float(len(seq))
        shannon_entropy_value -= p_i * math.log2(p_i)
 
    return round(shannon_entropy_value, 3)

def shannon_dimer_entropy(seq:str, k=2) -> float:
    seq = seq.upper()
    kmers = [seq[i:i+k] for i in range(len(seq) - k + 1)]
    counts = Counter(kmers)
    counts_sum = sum(counts.values())
    
    kmer_entropy_value = 0
    for c in counts.values():
        p_i = c / counts_sum
        kmer_entropy_value -= p_i * math.log2(p_i)
    
    return round(kmer_entropy_value, 3)

def calculate_repeats(seq: str) -> tuple[float]:
    seq = seq.upper()
    n = len(seq)

    sum_lens = {1: 0, 2: 0, 3: 0}

    for k in (1, 2, 3):
        i = 0
        while i <= n - k:
            motif = seq[i:i+k]
            if any(c not in "ACGT" for c in motif):
                i += 1
                continue

            if seq[i+k:i+2*k] != motif:
                i += 1
                continue

            j = i + k
            repeat_len = k
            while j <= n - k and seq[j:j+k] == motif:
                j += k
                repeat_len += k

            valid = False
            if k == 1:
                valid = repeat_len >= 5
            else:
                valid = repeat_len >= 3 * k

            if valid:
                sum_lens[k] += repeat_len
                i = j
            else:
                i += 1

    mono_sum = round(sum_lens[1] / n, 3)
    di_sum   = round(sum_lens[2] / n, 3)
    tri_sum  = round(sum_lens[3] / n, 3)

    total_sum = round((sum_lens[1] + sum_lens[2] + sum_lens[3]) / n, 3)

    return mono_sum, di_sum, tri_sum, total_sum


def calculate_repeat_masked_percent(seq:str) -> float:
    masked = sum(1 for base in seq if base.islower() or base == 'N')
    return round(masked / len(seq), 3)

Off-target Occurence Count

In [32]:
def get_prefixes(seq: str, kmer_len: int, olig_len: int) -> list[str]:
    """
    Эта функция составляет список префиксов последовательности
    :param seq: Последовательность, минимальная длина k-mer, максимальная длина k-mer (равная длине олигонуклеотида) 
    :return: Список префиксов
    """
    L = len(seq)
    prefixes = []

    for j in range(L - kmer_len + 1):
        if j+olig_len <= L:
            subseq = seq[j:j+olig_len]
            prefixes += [subseq[:i] for i in range(kmer_len, olig_len+1)]
        else:
            subseq = seq[j:L]
            prefixes += [subseq[:i] for i in range(kmer_len, L-j+1)]

    return prefixes

def kmer_counter(prefixes: list[str], fmindex: DNAFMIndex.Index, threads: int) -> int:
    """
    Эта функция рассчитывает количество 100%-ных совпадений префиксов последовательности в геноме
    :param seq: Последовательность
    :param prefixes: Список префиксов
    :param index: геномный индекс
    :param threads: количество ядер
    :return: Количество 100% off-target для префиксов
    """
    kmer_search_list = DNAFMIndex.KmerSearchList(len(prefixes)*2) # KmerSearchList capacity
    kmer_search_list.fill(prefixes)
    kmer_search_list.parallel_search_locate(fmindex, threads)
    
    offtarget_sum = 0
    for i in range(kmer_search_list.count):
        if kmer_search_list.kmer_search_data[i].count > 1: # count only off-target
            prefix_count = kmer_search_list.kmer_search_data[i].count
            offtarget_sum += prefix_count

    return offtarget_sum

In [33]:
def complement(seq:str) -> str:
    compl = str.maketrans('ACGTacgt', 'TGCAtgca')
    return seq.translate(compl)[::-1]

In [34]:
def get_stats(seq):
    gc = gc_content(seq)
    monomer_entropy = shannon_entropy(seq)
    dimer_entropy = shannon_dimer_entropy(seq)
    masked = calculate_repeat_masked_percent(seq)
    mono_sum, di_sum, tri_sum, total_sum = calculate_repeats(seq)

    return gc, monomer_entropy, dimer_entropy, masked, mono_sum, di_sum, tri_sum, total_sum

Example data. 

Overlap - the number of nucleotides overlapping between adjacent probes in the panel.

Off-target - off-target occurence count calculated using DNAFMIndex. Computation requires building an FM-index from the reference genome.

'Overlap' and 'Off-target' are predefined in this toy-example.

In [ ]:
data = [
    {'Seq': 'AACTGATCGCTGCATCTTCTTTTCGCGCAGGCGCATGCCTCTACGTCGCTAGCTCGACTGCT', 'Overlap': 0, 'Off-target': 10},
    {'Seq': 'GTCGCGCTCTATCTCTCTTGCGGGGCGCTATCTCTTGCTACGCTTCGATCGGCTGACTTCGC', 'Overlap':25, 'Off-target': 15},
    {'Seq': 'TTCGCTATTCTGCTCGGCGATGCATTAAAAAACGTGCATGCGTGGCTATGCTTCTCGTACTG', 'Overlap':25, 'Off-target': 0},
    {'Seq': 'ATGCTCGTCAATATATATCGATCTACTATCTACTATCATCAAAAACTGATCATCTACTTCGC', 'Overlap':10, 'Off-target': 0},
    {'Seq': 'GCGTCTTTACTGAAAACTGACATATATCTCATCGTACTACTATTATCTCTGCTCTCGTCTCG', 'Overlap':10, 'Off-target': 120},
]

test_df = pd.DataFrame(data)
test_probes = test_df.Seq

In [37]:
probes_seq_features = {}
for seq in test_probes:
    probes_seq_features[seq] = get_stats(seq)

df_seq_based = pd.DataFrame.from_dict(probes_seq_features, orient='index')
df_seq_based = df_seq_based.reset_index()
df_seq_based.columns = [
    'Seq', 'GC-content', 'Masked nucleotides fraction',
    'Shannon entropy of monomers', 'Shannon entropy of dimers',
    'Mononucleotide repeat fraction', 'Dinucleotide repeat fraction', 
    'Trinucleotide repeat fraction', 'Sum of repeat fractions',
]

df = test_df.merge(df_seq_based, on='Seq')

Thermodynamic Features

In [38]:
hybr_model = nupack.Model(material='dna04.2', sodium=0.8, magnesium=0, celsius=65)

targets = [complement(seq) for seq in test_probes]
probe_strands = [nupack.Strand(seq, name=f'probe_{i}') for i, seq in enumerate(test_probes)]
target_strands = [nupack.Strand(seq, name=f'target_{i}') for i, seq in enumerate(targets)]

hairpins = [nupack.Complex([p], name=f'hairpin_{i}') for i, p in enumerate(probe_strands)]
homodimers = [nupack.Complex([p, p], name=f'homodimer_{i}') for i, p in enumerate(probe_strands)]
ontargets = [nupack.Complex([p, t], name=f'ontarget_{i}') for i, (p, t) in enumerate(zip(probe_strands, target_strands))]
include_complexes = hairpins + homodimers + ontargets

results=nupack.complex_analysis(complexes=include_complexes, 
                                compute=['pfunc', 'mfe'], model=hybr_model)

In [39]:
probes_thermo_features = {}
for i in range(len(test_probes)):
    thermo_features=[]
    for complex in (hairpins[i], homodimers[i], ontargets[i]):
        r = results[complex]
        thermo_features.extend([round(r.free_energy, 2),
                                round(r.mfe[0].energy, 2)])
    probes_thermo_features[f'{test_probes[i]}'] = thermo_features

In [40]:
df_thermodynamics = pd.DataFrame.from_dict(probes_thermo_features, orient='index')
df_thermodynamics = df_thermodynamics.reset_index()
df_thermodynamics.columns = ['Seq', 'dG_hairpin', 'MFE_hairpin',
                       'dG_homodimer', 'MFE_homodimer',
                       'dG_ontarget', 'MFE_ontarget']

df = df.merge(df_thermodynamics, on='Seq')

In [41]:
df

,Seq,Overlap,Off-target,GC-content,Masked nucleotides fraction,Shannon entropy of monomers,Shannon entropy of dimers,Mononucleotide repeat fraction,Dinucleotide repeat fraction,Trinucleotide repeat fraction,Sum of repeat fractions,dG_hairpin,MFE_hairpin,dG_homodimer,MFE_homodimer,dG_ontarget,MFE_ontarget
0,AACTGATCGCTGCATCTTCTTTTCGCGCAGGCGCATGCCTCTACGT...,0,10,0.565,1.936,3.603,0.0,0.000,0.000,0.0,0.000,-4.10,-1.73,-15.52,-11.20,-54.94,-54.07
1,GTCGCGCTCTATCTCTCTTGCGGGGCGCTATCTCTTGCTACGCTTC...,25,15,0.597,1.853,3.258,0.0,0.000,0.097,0.0,0.097,-3.43,-1.27,-13.87,-10.03,-53.90,-52.95
2,TTCGCTATTCTGCTCGGCGATGCATTAAAAAACGTGCATGCGTGGC...,25,0,0.484,1.976,3.629,0.0,0.097,0.097,0.0,0.194,-3.52,-1.11,-14.09,-8.45,-49.46,-47.63
3,ATGCTCGTCAATATATATCGATCTACTATCTACTATCATCAAAAAC...,10,0,0.339,1.853,3.347,0.0,0.081,0.129,0.0,0.210,-2.93,0.00,-12.79,-6.23,-35.63,-34.91
4,GCGTCTTTACTGAAAACTGACATATATCTCATCGTACTACTATTAT...,10,120,0.403,1.909,3.454,0.0,0.000,0.097,0.0,0.097,-2.32,0.00,-11.47,-4.59,-38.55,-37.88
